In [28]:
from openai import OpenAI
import os
import re
from pathlib import Path

In [29]:
client = OpenAI(
    api_key= os.environ["TOKEN_KEY"],
    base_url="https://api.opentyphoon.ai/v1"
)

In [ ]:
def generate2csv(district, sub_district, unit_name , province="ลำปาง", content="", file_name=""):
    messages = [
        {"role": "system", "content": "This is a structured English instruction prompt designed for an AI model (like Gemini or GPT-4) to accurately parse your election data.\n\nPrompt Instruction for AI\nRole: You are a specialized Data Extraction Expert. Your task is to extract information from Thai Election Report OCR text and convert it into two specific CSV formats.\n\nData Source Analysis:\n\nThe input text describes either \"แบบบัญชีรายชื่อ\" (Party List)—labeled as \"บช\"—or \"แบบแบ่งเขตเลือกตั้ง\" (Constituency)—labeled as \"เขต\".\n\nLocation data follows keywords: จังหวัด (Province), อำเภอ (District), ตำบล (Sub-district), and หน่วยที่ (Unit).\n\nStrict Extraction Rules:\n\nType Identification: - If the text contains \"บัญชีรายชื่อ\" or \"(บช)\", type = บช.\n\nIf the text contains \"แบ่งเขตเลือกตั้ง\", type = เขต.\n\nValue Formatting:\n\nIf a specific field is missing from the text or the number is incorrect such as \"10-\", use -1. \n\nIf the value is explicitly zero, use 0.\n\nFor scores and ballot counts, prioritize Arabic numerals found in the text.\n\nLocation Cleanup: Remove the prefix words (จังหวัด, อำเภอ, ตำบล, หน่วยที่) and keep only the names/numbers.\n\nOutput Format 1: Results Table (CSV)\nStructure: type, province, district, sub-district, unit_number, name, score\n\nname: For \"บช\", use the Party Name (ชื่อพรรคการเมือง). For \"เขต\", use the Candidate Name (ชื่อตัว-ชื่อสกุล).\n\nscore: The numeric score obtained.\n\nOutput Format 2: Summary Table (CSV)\nStructure: type, province, district, sub-district, unit_number, total_ballots, valid_ballots, invalid_ballots, no_vote_ballots\n\ntotal_ballots: Total received (ได้รับบัตรเลือกตั้งทั้งหมด).\n\nvalid_ballots: บัตรดี.\n\ninvalid_ballots: บัตรเสีย.\n\nno_vote_ballots: บัตรไม่เลือกผู้สมัคร/พรรคใด.\n\nExample of Expected Output based on your first text:\nCSV 1:\n\ntype, province, district, sub-district, unit_number, name, score\nบช, ลำปาง, สำราญ, ท่าศาลา, 1, ไทยทรัพย์ทวี, 0\nบช, ลำปาง, สำราญ, ท่าศาลา, 1, เพื่อชาติไทย, 6\n\nCSV 2:\ntype, province, district, sub-district, unit_number, total_ballots, valid_ballots, invalid_ballots, no_vote_ballots\nบช, ลำปาง, สำราญ, ท่าศาลา, 1, 700, 663, 8, 29"},
        {"role": "user", "content": f"""จังหวัด: {province}, district: {district}, sub-district: {sub_district}, unit name: {unit_name}\n
        {content}\n
    """}
    ]
    stream = client.chat.completions.create(
        model="typhoon-v2.5-30b-a3b-instruct",
        messages=messages,
        temperature=0.5,
        max_completion_tokens=15000,
        top_p=0.6,
        frequency_penalty=0,
        stream=True
    )
    print(f"Processing CSVs: {file_name}")
    full_content = ""
    for chunk in stream:
        if chunk.choices[0].delta.content is not None:
            text = chunk.choices[0].delta.content
            full_content += text
    print(f"Success CSVs: {file_name}")
    csv_blocks = re.findall(r'```csv\n(.*?)\n```', full_content, re.DOTALL)
    if len(csv_blocks) >= 2:
        return (csv_blocks[0].strip(), csv_blocks[1].strip())
    else:
        return (None, None)


In [31]:
source_dir = Path("ocr-result")
files = sorted(list(source_dir.glob("**/*.txt")))
files

[WindowsPath('ocr-result/นอกเขต/ชุดที่ 1/ชุดที่ 1 ส.ส. 5-17(บช).txt'),
 WindowsPath('ocr-result/นอกเขต/ชุดที่ 1/ชุดที่ 1 ส.ส. 5-17.txt'),
 WindowsPath('ocr-result/นอกเขต/ชุดที่ 10/ชุดที่ 10 5-17(บช).txt'),
 WindowsPath('ocr-result/นอกเขต/ชุดที่ 10/ชุดที่ 10 5-17.txt'),
 WindowsPath('ocr-result/นอกเขต/ชุดที่ 11/ชุดที่ 11 5-17(บช).txt'),
 WindowsPath('ocr-result/นอกเขต/ชุดที่ 11/ชุดที่ 11 5-17.txt'),
 WindowsPath('ocr-result/นอกเขต/ชุดที่ 12/ชุดที่ 12 5-17 (บช).txt'),
 WindowsPath('ocr-result/นอกเขต/ชุดที่ 12/ชุดที่ 12 5-17.txt'),
 WindowsPath('ocr-result/นอกเขต/ชุดที่ 13/ชุดที่ 13 5-17 (บช).txt'),
 WindowsPath('ocr-result/นอกเขต/ชุดที่ 13/ชุดที่ 13 5-17.txt'),
 WindowsPath('ocr-result/นอกเขต/ชุดที่ 2/ชุดที่ 2 ส.ส. 5-17 (บช).txt'),
 WindowsPath('ocr-result/นอกเขต/ชุดที่ 2/ชุดที่ 2 ส.ส. 5-17.txt'),
 WindowsPath('ocr-result/นอกเขต/ชุดที่ 3/ชุดที่ 3 ส.ส. 5-17 (บช).txt'),
 WindowsPath('ocr-result/นอกเขต/ชุดที่ 3/ชุดที่ 3 ส.ส. 5-17.txt'),
 WindowsPath('ocr-result/นอกเขต/ชุดที่ 4/ชุดที่ 4 ส.ส.

In [32]:
def extract_file_info(filename):
    sub_district_match = re.search(r'(?:ตำบล|ต\.)\s*([\u0E00-\u0E7F]+)', filename)
    sub_district = sub_district_match.group(1) if sub_district_match else "-1"

    unit_match = re.search(r'หน่วยที่_?(\d+)', filename)
    unit_number = unit_match.group(1) if unit_match else "-1"

    return sub_district, unit_number

In [ ]:
for file in files:
    with open("except_extract.txt", "r", encoding="utf-8") as fr:
        line_checks = [line.strip() for line in fr]
    if file.stem in line_checks:
        print(f"file: {file.stem} already extract!")
        continue
    with open(file, 'r', encoding="utf-8") as f:
        lines = [line.strip() for line in f]
    content = "\n".join(lines) 
    structure = file.parent.parts
    sub_district, unit_num = extract_file_info(file.stem)
    if sub_district == "-1":
        sub_district = structure[2]
    csv1, csv2 = generate2csv(structure[1], sub_district, unit_num , province="ลำปาง", content=content, file_name=f"{file.stem}.txt")
    header1 = "type,province,district,sub-district,unit_number,name,score"
    header2 = "type,province,district,sub-district,unit_number,total_ballots,valid_ballots,invalid_ballots,no_vote_ballots"

    csv1_clean = csv1.split(header1)[-1].strip()
    csv2_clean = csv2.split(header2)[-1].strip()
    print(f"Starting save result of file: {file.stem}")
    # print(csv1_clean)
    # print("----------------------")
    # print(csv2_clean)
    with open(f"master_results.csv", "a", encoding="utf-8") as fw1:
        fw1.write(csv1_clean+"\n")
    with open(f"master_summary.csv", "a", encoding="utf-8") as fw2:
        fw2.write(csv2_clean+"\n")
    with open(f"except_extract.txt", "a", encoding="utf-8") as fw3:
        fw3.write(f"{file.stem}\n")
    print(f"Success save result on file: {file.stem}")

Processing CSVs: ชุดที่ 1 ส.ส. 5-17(บช).txt
Success CSVs: ชุดที่ 1 ส.ส. 5-17(บช).txt
บช,ลำปาง,นอกเขต,-1,-1,ไทยทรัพย์ทวี,0
บช,ลำปาง,นอกเขต,-1,-1,เพื่อชาติไทย,6
บช,ลำปาง,นอกเขต,-1,-1,ใหม่,4
บช,ลำปาง,นอกเขต,-1,-1,มิติใหม่,0
บช,ลำปาง,นอกเขต,-1,-1,รวมใจไทย,1
บช,ลำปาง,นอกเขต,-1,-1,รวมไทยสร้างชาติ,9
บช,ลำปาง,นอกเขต,-1,-1,พลวัต,1
บช,ลำปาง,นอกเขต,-1,-1,ประชาธิปไตยใหม่,1
บช,ลำปาง,นอกเขต,-1,-1,เพื่อไทย,67
บช,ลำปาง,นอกเขต,-1,-1,ทางเลือกใหม่,3
บช,ลำปาง,นอกเขต,-1,-1,เศรษฐกิจ,8
บช,ลำปาง,นอกเขต,-1,-1,เสรีรวมไทย,0
บช,ลำปาง,นอกเขต,-1,-1,รวมพลังประชาชน,4
บช,ลำปาง,นอกเขต,-1,-1,ท้องที่ไทย,0
บช,ลำปาง,นอกเขต,-1,-1,อนาคตไทย,1
บช,ลำปาง,นอกเขต,-1,-1,พลังเพื่อไทย,1
บช,ลำปาง,นอกเขต,-1,-1,ไทยชนะ,0
บช,ลำปาง,นอกเขต,-1,-1,พลังสังคมใหม่,1
บช,ลำปาง,นอกเขต,-1,-1,สังคมประชาธิปไตยไทย,0
บช,ลำปาง,นอกเขต,-1,-1,พิวซัน,0
บช,ลำปาง,นอกเขต,-1,-1,ไทรวมพลัง,0
บช,ลำปาง,นอกเขต,-1,-1,ก้าวล่วง,0
บช,ลำปาง,นอกเขต,-1,-1,ปวงชนไทย,0
บช,ลำปาง,นอกเขต,-1,-1,วิชชันใหม่,0
บช,ลำปาง,นอกเขต,-1,-1,เพื่อชีวิตใหม่,0
บช,ลำปาง,นอกเขต,-1,-1,คลองไทย,0
บช,